# G12: Deletion Tuning on a quantized base (Kaggle)

Does G10's Deletion Tuning (mask over `gate_proj` weights, official recipe defaults) work on a Q8_0 or Q4_0 base as well
as on the unquantized model, and what does the quantized base save in memory? Arms: `base`, `base_q8_0`, `base_q4_0`
(no DT), `dt`, `dt_q8_0`, `dt_q4_0`, and `dt_then_q8_0` / `dt_then_q4_0` (DT first, quantize the baked weights after;
computed inside the `dt` run). Metric: held-out wikitext-2 NLL per token, with paired intervals over windows.
Details: `teams/12-quantization/dt_quant/dt_on_quantized.py` docstring. `decoder_weights_GB` and `dt_state_GB` are storage figures;
`peak_cuda_GB` is measured during training and need not fall with the format. `prior_deleted` must be above 0 on every DT
arm (weights that the format rounded to zero are kept out of the Wanda prior's threshold).

**Smoke run first:** keep `STEPS = 4`, `TRAIN_WINDOWS = 16`, `TEST_WINDOWS = 8` and `ARMS = ["base", "dt_q8_0"]`; check that
both arms print an NLL. Then set the real values. Kaggle: Accelerator **GPU T4 x2**, Internet **on**. No token needed.

In [ ]:
# --- settings -----------------------------------------------------------------------------------------------
RUN_BY        = ""                      # <-- your GitHub handle, recorded with the results
MODEL         = "Qwen/Qwen3-0.6B-Base"
REVISION      = None                    # <-- pin a commit of the model before the real run and write it in the report
ARMS          = ["base", "dt_q8_0"]     # real run: ["base","base_q8_0","base_q4_0","dt","dt_q8_0","dt_q4_0","dt_then_q8_0","dt_then_q4_0"]
STEPS         = 4                       # real run: 400
TRAIN_WINDOWS = 16                      # real run: 2000 (windows of LENGTH tokens)
TEST_WINDOWS  = 8                       # real run: 200
LENGTH        = 512
BATCH, MICRO  = 8, 2                    # rows per step, rows per forward (lower MICRO if out of memory)
LR            = 5e-4                    # G10's DT default since its 17 Nov port check (older runs: 2e-3); label any other value
SEED          = 1
TORCH         = "torch==2.11.0"         # CUDA 12.6 build, from the PyTorch index below
TORCH_INDEX   = "https://download.pytorch.org/whl/cu126"
NOTEBOOK_VERSION = "38dd373945f2"

In [ ]:
import datetime, glob, json, os, shutil, subprocess, time
OUT = "/kaggle/working/dt_quant_out"; WORK = "/kaggle/tmp/dt_quant_work"; CODE = f"{WORK}/code"
for d in (OUT, f"{OUT}/logs", WORK, CODE):
    os.makedirs(d, exist_ok=True)

def sh(cmd, log=None):
    print("$", cmd, flush=True)
    full = f"set -o pipefail; ( {cmd} ) 2>&1" + (f" | tee -a {log}" if log else "")
    rc = subprocess.run(full, shell=True, executable="/bin/bash").returncode
    if rc != 0:
        raise RuntimeError(f"command failed (exit {rc}): {cmd}" + (f"; see {log}" if log else ""))

sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")
sh("curl -fsS -o /dev/null https://huggingface.co || (echo 'NO INTERNET: turn it on in the notebook settings' && false)")
ENV = {"started": datetime.datetime.now().isoformat(timespec="seconds"), "run_by": RUN_BY, "notebook_version": NOTEBOOK_VERSION,
       "model": MODEL, "revision": REVISION}

In [ ]:
# the code, embedded by build_notebook.py (mask.py is G10's dt/mask.py, unchanged)
SOURCES = {
"dt_on_quantized.py": "#!/usr/bin/env python3\n\"\"\"Deletion Tuning on a quantized base model (G12 design task 9, with G10's dt/mask.py).\n\n    python dt_on_quantized.py --model Qwen/Qwen3-0.6B-Base --out dt_quant.json \\\\\n        [--arms base,base_q8_0,base_q4_0,dt,dt_q8_0,dt_q4_0,dt_then_q8_0,dt_then_q4_0] [--steps 400] [--device cuda]\n\nTSAI's question: does Deletion Tuning work on a quantized base as well as on bf16, and what does the quantized base save\nin memory? DT freezes W and trains one FP32 logit G per targeted weight; the model applies W * 1[G > 0]. Here W can be\nstored quantized: Q8_0 or Q4_0, the llama.cpp block formats (blocks of 32 along the input dimension, one fp16 scale per\nblock; same values as gguf-py, tested), kept packed on the targeted layers and dequantized in the forward pass. Every\nother decoder linear is quantized in place (round-trip through the format). Embeddings, head and norms stay as loaded.\n\nArms (names; \"<q>\" is q8_0 or q4_0):\n  base, base_<q>      no DT: the dense model, and the dense model with every decoder linear quantized\n  dt                  DT on the unquantized base\n  dt_<q>              DT on the quantized base: W packed, the Wanda prior calibrated on the quantized weights\n  dt_then_<q>         the `dt` result with its mask baked in (W * 1[G > 0]), then quantized: the other order, as a control\n                      (computed inside the `dt` run, which bakes the mask once and scores a quantized copy per format)\nWith --grad-check it also runs the instructor's gradient-coverage check (28 Nov): one forward and backward with the packed path on and off,\nlisting every trainable parameter whose gradient is None or all zero, and comparing the two lists and the gradients.\nEach arm reports held-out NLL per token (mean over windows, with per-window values for paired intervals), the share of\ntargeted weights deleted, training time and peak CUDA memory. Training uses G10's DT settings: AdamW on the logits (lr 5e-4, G10's default since its\n17 Nov port check; its older ladder runs used 2e-3, set with --lr; betas 0.9/0.95), alpha 100 to 1000, Wanda prior (eta 1/2048, strength 3),\nregulariser lambda 1e-5 decaying to 0, `gate_proj` targets. The loop is simpler than G10's trainer (sft/bench/train.py): constant\nlearning rate, no warmup, no cosine decay, no gradient clipping. That is the same for every arm, so arms compare fairly, but\nthe numbers are not G10's trainer's. The data is plain text (wikitext-2), next-token loss on every token:\nnot G10's SFT slice, so this answers \"does the mask train on a quantized base\", not G10's chat-format question.\n\nMemory: `memory_accounting` gives bytes per decoder-linear weight (fp32 4, bf16 2, Q8_0 34/32, Q4_0 18/32) and the DT state per\ntargeted weight (FP32 logit 4 + gradient 4 + Adam moments 8 = 16 bytes; DT arms only), so the table shows how much of DT's cost\nthe quantized base can touch. These are STORAGE figures. Peak CUDA memory during training is measured separately and need not\nfall with the format: `base_weight` dequantizes on every read and `W * mask` keeps that fp32 copy for backward in every layer\n(Sourav measured 11.9 against 7.9 bytes per masked weight saved for backward at Q8_0), so a packed arm can peak as high as `dt`.\n`dtmask.deletion_stats()` counts `named_parameters()`, and packing removes `weight`, so its `model_weights` is lower on packed\nmodels; no number reported here uses it. Aggregate numbers only: no text is written.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse, importlib.util, json, math, sys, time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\n\nHERE = Path(__file__).resolve().parent\n_spec = importlib.util.spec_from_file_location(\"g10_dt_mask\", HERE.parents[1] / \"10-post-training\" / \"dt\" / \"mask.py\")\ndtmask = importlib.util.module_from_spec(_spec); sys.modules[\"g10_dt_mask\"] = dtmask; _spec.loader.exec_module(dtmask)\n\nBLOCK = 32\nBYTES_PER_WEIGHT = {\"bf16\": 2.0, \"bfloat16\": 2.0, \"float32\": 4.0, \"q8_0\": 34 / 32, \"q4_0\": 18 / 32}   # llama.cpp block sizes: 2 B scale + 32 / 16 B codes\nDT_STATE_BYTES = 4 + 4 + 8                                          # FP32 logit, its gradient, Adam exp_avg and exp_avg_sq\n\n\n# ------------------------------------------------------------------------------------------------ block quantization\ndef _blocks(w: torch.Tensor) -> torch.Tensor:\n    if w.shape[-1] % BLOCK:\n        raise ValueError(f\"input dimension {w.shape[-1]} is not a multiple of {BLOCK}\")\n    return w.float().reshape(*w.shape[:-1], w.shape[-1] // BLOCK, BLOCK)\n\n\ndef _round_away(x: torch.Tensor) -> torch.Tensor:\n    return torch.sign(x) * torch.floor(x.abs() + 0.5)           # C roundf: halves away from zero\n\n\ndef quantize(w: torch.Tensor, fmt: str) -> tuple[torch.Tensor, torch.Tensor]:\n    \"\"\"llama.cpp's reference quantizers. Returns (codes, scales): Q8_0 int8 [.., nb, 32]; Q4_0 uint8 packed [.., nb, 16]\n    (element j in the low nibble, j + 16 in the high one); scales fp16 [.., nb].\"\"\"\n    b = _blocks(w)\n    if fmt == \"q8_0\":\n        d = b.abs().amax(-1) / 127.0\n        inv = torch.where(d > 0, 1.0 / d, torch.zeros_like(d))\n        return _round_away(b * inv[..., None]).to(torch.int8), d.to(torch.float16)\n    if fmt == \"q4_0\":\n        idx = b.abs().argmax(-1, keepdim=True)\n        mx = b.gather(-1, idx).squeeze(-1)                       # the signed value with the largest magnitude\n        d = mx / -8.0\n        inv = torch.where(d != 0, 1.0 / d, torch.zeros_like(d))\n        q = torch.clamp(torch.floor(b * inv[..., None] + 8.5), max=15).to(torch.uint8)\n        return (q[..., :16] | (q[..., 16:] << 4)).contiguous(), d.to(torch.float16)\n    raise ValueError(fmt)\n\n\ndef dequantize(codes: torch.Tensor, scales: torch.Tensor, fmt: str, shape, dtype) -> torch.Tensor:\n    d = scales.float()[..., None]\n    if fmt == \"q8_0\":\n        x = codes.float() * d\n    else:\n        lo, hi = (codes & 0x0F).float(), (codes >> 4).float()\n        x = (torch.cat([lo, hi], dim=-1) - 8.0) * d\n    return x.reshape(shape).to(dtype)\n\n\ndef fake_quantize_(linear: nn.Linear, fmt: str) -> None:\n    \"\"\"Round-trip the weight through the format, stored back in its own dtype.\"\"\"\n    c, s = quantize(linear.weight.data, fmt)\n    linear.weight.data.copy_(dequantize(c, s, fmt, linear.weight.shape, linear.weight.dtype))\n\n\nclass QuantMaskedLinear(dtmask.MaskedLinear):\n    \"\"\"A MaskedLinear whose frozen W is stored packed (Q8_0 or Q4_0) and dequantized on every read of `base_weight`.\n    Built by `to_packed` from a MaskedLinear; `weight`, `forward`, the Wanda hooks, the regulariser and `bake` of\n    mask.py all read `base_weight`, so they work unchanged.\"\"\"\n    fmt: str\n\n    @property\n    def base_weight(self) -> torch.Tensor:\n        return dequantize(self.codes, self.scales, self.fmt, (self.out_features, self.in_features), self._wdtype)\n\n    def packed_bytes(self) -> int:\n        return self.codes.numel() * self.codes.element_size() + self.scales.numel() * self.scales.element_size()\n\n\ndef to_packed(m: \"dtmask.MaskedLinear\", fmt: str) -> \"QuantMaskedLinear\":\n    w = m._parameters.pop(\"weight\")\n    codes, scales = quantize(w.data, fmt)\n    m.__class__ = QuantMaskedLinear\n    m.fmt, m._wdtype = fmt, w.dtype\n    m.register_buffer(\"codes\", codes.to(w.device)); m.register_buffer(\"scales\", scales.to(w.device))\n    return m\n\n\n# ------------------------------------------------------------------------------------------------ model helpers\ndef decoder_linears(model: nn.Module) -> list[tuple[str, nn.Linear]]:\n    return [(n, m) for n, m in model.named_modules() if isinstance(m, nn.Linear) and \".layers.\" in f\".{n}\"]\n\n\ndef quantize_decoder_(model: nn.Module, fmt: str, skip=()) -> int:\n    n = 0\n    for name, lin in decoder_linears(model):\n        if not (skip and dtmask.target_match(name, skip)):\n            fake_quantize_(lin, fmt); n += 1\n    return n\n\n\ndef memory_accounting(model: nn.Module, targets, fmt: str) -> dict:\n    \"\"\"Bytes of the decoder linears under `fmt` plus DT's per-weight state, for the model as loaded (no CUDA needed).\"\"\"\n    lins = decoder_linears(model)\n    total = sum(m.weight.numel() for _, m in lins)\n    masked = sum(m.weight.numel() for n, m in lins if targets and dtmask.target_match(n, targets))\n    per_w = BYTES_PER_WEIGHT[fmt]\n    return {\"decoder_linear_weights\": total, \"masked_weights\": masked, \"format\": fmt,\n            \"weights_bytes\": round(total * per_w), \"dt_state_bytes\": masked * DT_STATE_BYTES,\n            \"total_bytes\": round(total * per_w) + masked * DT_STATE_BYTES,\n            \"bf16_weights_bytes\": total * 2, \"saved_by_format_bytes\": round(total * (2 - per_w))}\n\n\n# ------------------------------------------------------------------------------------------------ data and scoring\ndef make_windows(token_ids: list[int], length: int, n: int, offset: int = 0) -> list[list[int]]:\n    out = []\n    for i in range(n):\n        s = offset + i * length\n        if s + length > len(token_ids):\n            break\n        out.append(token_ids[s:s + length])\n    return out\n\n\n@torch.no_grad()\ndef window_nll(model: nn.Module, windows: list[list[int]], device, batch: int = 4) -> list[float]:\n    \"\"\"Mean NLL per predicted token of every window, in eval mode (DT layers apply the hard mask).\"\"\"\n    model.eval()\n    out = []\n    for i in range(0, len(windows), batch):\n        ids = torch.tensor(windows[i:i + batch], device=device)\n        logits = model(input_ids=ids).logits[:, :-1].float()\n        nll = nn.functional.cross_entropy(logits.reshape(-1, logits.shape[-1]), ids[:, 1:].reshape(-1), reduction=\"none\")\n        out += nll.reshape(ids.shape[0], -1).mean(1).tolist()\n    return out\n\n\ndef paired_bootstrap(a: list[float], b: list[float], n_boot: int = 2000, seed: int = 0) -> dict:\n    \"\"\"Mean of (a - b) over the same windows with a 95% percentile interval (resampling windows).\"\"\"\n    d = np.asarray(a) - np.asarray(b)\n    rng = np.random.default_rng(seed)\n    means = d[rng.integers(0, len(d), (n_boot, len(d)))].mean(1)\n    lo, hi = np.percentile(means, [2.5, 97.5])\n    return {\"mean_diff\": float(d.mean()), \"ci95\": [float(lo), float(hi)], \"windows\": int(len(d))}\n\n\n# ------------------------------------------------------------------------------------------------ DT training\ndef run_arm(arm: str, args, tok_windows: dict, device) -> dict:\n    \"\"\"One arm end to end. arm is base | base_<q> | dt | dt_<q> | dt_then_<q>.\"\"\"\n    from transformers import AutoModelForCausalLM\n    kind = \"base\" if arm.startswith(\"base\") else \"dt\"\n    fmt = arm.split(\"_\", 1)[1] if \"_\" in arm else None\n    assert not arm.startswith(\"dt_then\"), \"dt_then_<q> comes from the dt arm (it bakes and quantizes the same run)\"\n    assert fmt in (None, \"q8_0\", \"q4_0\"), arm\n    if fmt and args.dtype != \"float32\":\n        raise ValueError(f\"{arm}: with --dtype {args.dtype} the dequantized values are rounded again and are no longer {fmt}; use float32\")\n    torch.manual_seed(args.seed)\n    model = AutoModelForCausalLM.from_pretrained(args.model, torch_dtype=getattr(torch, args.dtype), revision=args.revision).to(device)\n    targets = dtmask.TARGET_SETS[args.targets]\n    # the format's own bytes (the loaded dtype for an unquantized arm); DT state only where DT runs\n    res = {\"arm\": arm, \"format\": fmt or args.dtype, \"memory\": memory_accounting(model, targets if kind == \"dt\" else (), fmt or args.dtype)}\n    if device.type == \"cuda\":\n        torch.cuda.reset_peak_memory_stats()\n    t0 = time.time()\n    if kind == \"base\":\n        if fmt:\n            quantize_decoder_(model, fmt)\n    else:\n        train_fmt = fmt\n        if train_fmt:                                           # every non-targeted decoder linear is quantized in place\n            quantize_decoder_(model, train_fmt, skip=targets)\n        names = dtmask.convert(model, targets, args.alpha_start, args.alpha_end, args.steps, args.seed)\n        if train_fmt:\n            for _, m in dtmask.masked_layers(model):\n                to_packed(m, train_fmt)\n            res[\"packed_bytes_targeted\"] = sum(m.packed_bytes() for _, m in dtmask.masked_layers(model))\n            res[\"bf16_bytes_targeted\"] = sum(m.in_features * m.out_features * 2 for _, m in dtmask.masked_layers(model))\n        res[\"masked_layers\"] = len(names)\n        res.update(fit_mask(model, tok_windows[\"train\"], tok_windows[\"calib\"], device, args))\n    res[\"train_seconds\"] = round(time.time() - t0, 1)\n    if device.type == \"cuda\":\n        res[\"peak_cuda_gb\"] = round(torch.cuda.max_memory_allocated() / 1e9, 3)\n    nll = window_nll(model, tok_windows[\"test\"], device)\n    res.update(test_nll_mean=float(np.mean(nll)), test_windows=len(nll), per_window_nll=[round(x, 5) for x in nll])\n    if kind == \"dt\" and fmt is None and args.then:             # the other order: bake the hard mask, then quantize the baked weights\n        import copy\n        dtmask.bake(model)\n        res[\"then\"] = {}\n        for q in args.then:\n            clone = copy.deepcopy(model)\n            quantize_decoder_(clone, q)\n            nq = window_nll(clone, tok_windows[\"test\"], device)\n            res[\"then\"][q] = {\"arm\": f\"dt_then_{q}\", \"format\": q, \"memory\": memory_accounting(clone, (), q),   # served model: no DT state\n                              \"deletion_final\": res[\"deletion_final\"], \"test_nll_mean\": float(np.mean(nq)), \"test_windows\": len(nq),\n                              \"per_window_nll\": [round(x, 5) for x in nq]}\n            del clone\n    del model\n    return res\n\n\ndef fit_mask(model, train_windows, calib_windows, device, args) -> dict:\n    \"\"\"Wanda prior on the calibration windows, then `steps` optimizer steps on the logits (the official recipe's loop).\"\"\"\n    batches = [{\"input_ids\": torch.tensor([w], device=device), \"attention_mask\": torch.ones(1, len(w), dtype=torch.long, device=device)}\n               for w in calib_windows]\n    scores = dtmask.wanda_scores(model, batches, device)\n    # A weight that Q8_0 or Q4_0 rounded to exactly 0 has Wanda score 0. The prior marks the scores below the k-th smallest\n    # (k = eta x weights); with 0.8% (Q8_0) or 12% (Q4_0) of a layer at 0, that threshold is 0 and nothing is below it, so the\n    # quantized arms would train without the prior that `dt` has. Such weights get +inf: they are no candidates and never set\n    # the threshold. mask.py is untouched. (Found by Sourav's review of #889.)\n    zeroed = 0\n    for name, layer in dtmask.masked_layers(model):\n        if name in scores:\n            z = layer.base_weight == 0\n            zeroed += int(z.sum())\n            scores[name] = torch.where(z, torch.full_like(scores[name], float(\"inf\")), scores[name])\n    info = {\"wanda_prior\": dtmask.apply_wanda_prior(model, scores, args.eta, args.prior_strength), \"weights_zero_before_prior\": zeroed}\n    info[\"deletion_at_init\"] = dtmask.deletion_stats(model)[\"deleted_frac_of_targeted\"]\n    gates = [m.gate for _, m in dtmask.masked_layers(model)]\n    opt = torch.optim.AdamW(gates, lr=args.lr, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.0)\n    rng = np.random.default_rng(args.seed)\n    model.train()\n    losses = []\n    for step in range(args.steps):\n        idx = rng.choice(len(train_windows), args.batch, replace=len(train_windows) < args.batch)\n        total = 0.0\n        for j in range(0, args.batch, args.micro):\n            ids = torch.tensor([train_windows[i] for i in idx[j:j + args.micro]], device=device)\n            logits = model(input_ids=ids).logits[:, :-1].float()\n            loss = nn.functional.cross_entropy(logits.reshape(-1, logits.shape[-1]), ids[:, 1:].reshape(-1))\n            (loss * ids.shape[0] / args.batch).backward()\n            total += float(loss.detach()) * ids.shape[0] / args.batch\n        if args.lam > 0:\n            dtmask.regulariser_backward_(model, dtmask.lambda_at(step, args.steps, args.lam))\n        opt.step(); opt.zero_grad(set_to_none=True); dtmask.step_all(model)\n        losses.append(total)\n        if step % max(1, args.steps // 10) == 0:\n            print(f\"  step {step}/{args.steps} loss {total:.4f}\", flush=True)\n    stats = dtmask.deletion_stats(model)\n    info.update(train_loss_first=losses[0], train_loss_last=float(np.mean(losses[-10:])), deletion_final=stats[\"deleted_frac_of_targeted\"],\n                deleted=stats[\"deleted\"], targeted=stats[\"targeted\"], per_layer_min=stats[\"per_layer_min\"], per_layer_max=stats[\"per_layer_max\"])\n    return info\n\n\n# ------------------------------------------------------------------------------------------------ gradient coverage\ndef one_step_grads(model: nn.Module, windows: list, device, rows: int = 2) -> dict:\n    \"\"\"One forward and backward on the first `rows` windows: {name: grad clone or None} for every trainable parameter.\"\"\"\n    model.train()\n    ids = torch.tensor(windows[:rows], device=device)\n    logits = model(input_ids=ids).logits[:, :-1].float()\n    nn.functional.cross_entropy(logits.reshape(-1, logits.shape[-1]), ids[:, 1:].reshape(-1)).backward()\n    return {n: (p.grad.detach().clone() if p.grad is not None else None) for n, p in model.named_parameters() if p.requires_grad}\n\n\ndef coverage(grads: dict) -> dict:\n    \"\"\"Trainable parameters whose gradient is None or all zero, and the share of gradient elements that are exactly zero.\"\"\"\n    none = sorted(n for n, g in grads.items() if g is None)\n    zero = sorted(n for n, g in grads.items() if g is not None and not bool(g.abs().any()))\n    elems = sum(g.numel() for g in grads.values() if g is not None)\n    zeros = sum(int((g == 0).sum()) for g in grads.values() if g is not None)\n    return {\"trainable_parameters\": len(grads), \"grad_none\": none, \"grad_all_zero\": zero,\n            \"zero_element_share\": round(zeros / elems, 6) if elems else None}\n\n\ndef build_dt_model(args, fmt: str, packed: bool, device):\n    \"\"\"A DT model on a `fmt` base with fresh gates (same seed): the fast path (W packed) or the plain path (every decoder\n    linear fake-quantized in place, dense). No Wanda prior, so only the two paths differ.\"\"\"\n    from transformers import AutoModelForCausalLM\n    torch.manual_seed(args.seed)\n    model = AutoModelForCausalLM.from_pretrained(args.model, torch_dtype=getattr(torch, args.dtype), revision=args.revision).to(device)\n    targets = dtmask.TARGET_SETS[args.targets]\n    quantize_decoder_(model, fmt, skip=targets if packed else ())\n    dtmask.convert(model, targets, args.alpha_start, args.alpha_end, args.steps, args.seed)\n    if packed:\n        for _, m in dtmask.masked_layers(model):\n            to_packed(m, fmt)\n    return model\n\n\ndef grad_path_check(args, windows: list, device, fmt: str) -> dict:\n    \"\"\"Instructor's rule (28 Nov): a speed or memory path comes with a gradient-coverage check, fast path on and off.\n    On = W packed (Q8_0 or Q4_0); off = the same quantized values stored dense. Same data, same gates.\"\"\"\n    out = {}\n    grads = {}\n    for name, packed in ((\"fast_path_on\", True), (\"fast_path_off\", False)):\n        model = build_dt_model(args, fmt, packed, device)\n        grads[name] = one_step_grads(model, windows, device)\n        out[name] = coverage(grads[name])\n        del model\n    a, b = grads[\"fast_path_on\"], grads[\"fast_path_off\"]\n    same_names = sorted(a) == sorted(b)\n    out[\"same_trainable_parameters\"] = same_names\n    out[\"same_none_and_zero_lists\"] = (out[\"fast_path_on\"][\"grad_none\"] == out[\"fast_path_off\"][\"grad_none\"]\n                                       and out[\"fast_path_on\"][\"grad_all_zero\"] == out[\"fast_path_off\"][\"grad_all_zero\"])\n    out[\"max_abs_grad_difference\"] = (max(float((a[n] - b[n]).abs().max()) for n in a if a[n] is not None and b[n] is not None)\n                                      if same_names else None)\n    out[\"format\"] = fmt\n    return out\n\n\ndef load_text_windows(model_name: str, revision, length: int, n_train: int, n_test: int, n_calib: int) -> dict:\n    \"\"\"wikitext-2 (raw) from its parquet files (no `datasets` package: its huggingface-hub pin clashes with transformers 4.57).\"\"\"\n    import hashlib\n    import pyarrow.parquet as pq\n    from huggingface_hub import hf_hub_download\n    from transformers import AutoTokenizer\n    tok = AutoTokenizer.from_pretrained(model_name, revision=revision)\n    ids, sha = {}, {}\n    for split in (\"train\", \"test\"):\n        path = hf_hub_download(\"Salesforce/wikitext\", f\"wikitext-2-raw-v1/{split}-00000-of-00001.parquet\", repo_type=\"dataset\")\n        sha[split] = hashlib.sha256(Path(path).read_bytes()).hexdigest()\n        ids[split] = tok(\"\\n\\n\".join(pq.read_table(path).column(\"text\").to_pylist()), add_special_tokens=False)[\"input_ids\"]\n    return {\"train\": make_windows(ids[\"train\"], length, n_train), \"calib\": make_windows(ids[\"train\"], length, n_calib, offset=n_train * length),\n            \"test\": make_windows(ids[\"test\"], length, n_test), \"sha256\": sha}\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument(\"--model\", required=True); ap.add_argument(\"--revision\", default=None)\n    ap.add_argument(\"--out\", required=True, type=Path)\n    ap.add_argument(\"--arms\", default=\"base,base_q8_0,base_q4_0,dt,dt_q8_0,dt_q4_0,dt_then_q8_0,dt_then_q4_0\")\n    ap.add_argument(\"--targets\", default=\"gate\", choices=sorted(dtmask.TARGET_SETS))\n    ap.add_argument(\"--steps\", type=int, default=400); ap.add_argument(\"--batch\", type=int, default=8)\n    ap.add_argument(\"--micro\", type=int, default=2); ap.add_argument(\"--length\", type=int, default=512)\n    ap.add_argument(\"--train-windows\", type=int, default=2000); ap.add_argument(\"--calib-windows\", type=int, default=32)\n    ap.add_argument(\"--test-windows\", type=int, default=200)\n    ap.add_argument(\"--lr\", type=float, default=5e-4); ap.add_argument(\"--eta\", type=float, default=1 / 2048)\n    ap.add_argument(\"--prior-strength\", type=float, default=3.0); ap.add_argument(\"--lam\", type=float, default=1e-5)\n    ap.add_argument(\"--alpha-start\", type=float, default=100.0); ap.add_argument(\"--alpha-end\", type=float, default=1000.0)\n    ap.add_argument(\"--grad-check\", action=\"store_true\", help=\"after the arms: one forward and backward with the packed path on and off, \"\n                    \"listing trainable parameters whose gradient is None or all zero (written to results.json)\")\n    ap.add_argument(\"--dtype\", default=\"float32\", choices=[\"float32\", \"bfloat16\"])\n    ap.add_argument(\"--device\", default=\"cuda\" if torch.cuda.is_available() else \"cpu\"); ap.add_argument(\"--seed\", type=int, default=1)\n    a = ap.parse_args()\n    device = torch.device(a.device)\n    windows = load_text_windows(a.model, a.revision, a.length, a.train_windows, a.test_windows, a.calib_windows)\n    done = json.loads(a.out.read_text()) if a.out.exists() else {\"args\": {k: str(v) for k, v in vars(a).items()}, \"arms\": {}}\n    done[\"data_sha256\"] = windows[\"sha256\"]\n    wanted = a.arms.split(\",\")\n    a.then = [x[len(\"dt_then_\"):] for x in wanted if x.startswith(\"dt_then_\")]\n    for arm in [x for x in wanted if not x.startswith(\"dt_then_\")]:\n        if arm in done[\"arms\"]:\n            print(arm, \"already done\"); continue\n        print(\"==\", arm, flush=True)\n        r = done[\"arms\"][arm] = run_arm(arm, a, windows, device)\n        for q, sub in r.pop(\"then\", {}).items():\n            done[\"arms\"][f\"dt_then_{q}\"] = sub\n        print(f\"{arm}: NLL {r['test_nll_mean']:.4f}, deleted {r.get('deletion_final')}, {r['train_seconds']}s, peak {r.get('peak_cuda_gb')} GB\", flush=True)\n        a.out.write_text(json.dumps(done, indent=1))\n    if a.grad_check and \"grad_coverage\" not in done:\n        done[\"grad_coverage\"] = {q: grad_path_check(a, windows[\"train\"], device, q) for q in (\"q8_0\", \"q4_0\")}\n        for q, r in done[\"grad_coverage\"].items():\n            print(f\"grad coverage {q}: on {r['fast_path_on']['grad_none']} none / {r['fast_path_on']['grad_all_zero']} zero, off the same: \"\n                  f\"{r['same_none_and_zero_lists']}, max grad difference {r['max_abs_grad_difference']}\", flush=True)\n    arms = done[\"arms\"]\n    done[\"paired\"] = {f\"{x}-{y}\": paired_bootstrap(arms[x][\"per_window_nll\"], arms[y][\"per_window_nll\"])\n                      for x, y in ((\"dt\", \"base\"), (\"dt_q8_0\", \"base_q8_0\"), (\"dt_q4_0\", \"base_q4_0\"), (\"dt_q8_0\", \"dt\"),\n                                   (\"dt_q4_0\", \"dt\"), (\"dt_then_q8_0\", \"dt_q8_0\"), (\"dt_then_q4_0\", \"dt_q4_0\")) if x in arms and y in arms}\n    a.out.write_text(json.dumps(done, indent=1))\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\n",
"mask.py": "\"\"\"Deletion Tuning (DT) masks for linear layers: soft mask in training, hard mask for every evaluation.\n\nPort of the official implementation, agentic-learning-ai-lab/deleting-weights @ 94f0340ebb574f95bb41d844165253a6785e215e\n(MIT licence, copyright 2026 agentic learning ai lab): `src/learnable_sparsity/differentiable_mask.py`,\n`src/learnable_sparsity/sparse_linear.py`, `src/wanda_calibration.py`, and the regulariser of `src/train_sft.py`.\nPaper: arXiv 2611.10866v1, section 4 (eq. 5-8) and 5.1.\n\nWhat is the same as the official code\n- One FP32 logit G per targeted weight, N(0, 0.01^2), then the Wanda prior: G += sign * std(G) * prior_strength\n  (sign -1 for the lowest `eta` fraction of Wanda scores |W_ij| * ||X_j||_2 per layer, +1 otherwise). With\n  prior_strength 3 this is the paper's G ~ N(+-0.03, 0.01^2).\n- Training forward: W * sigmoid(alpha * G); evaluation forward: W * 1[G > 0]. alpha rises linearly from\n  alpha_start to alpha_end over `total_steps` optimizer steps (`step()` after each optimizer step).\n- Regulariser: loss -= lambda_t * sum ||W * soft_mask||^2 with lambda_t = lambda * (1 - t / T) (official train_sft.py\n  decays it to 0; the paper text states lambda = 1e-5 without the decay).\n- Targets are chosen by module-name substring and must be nn.Linear (official `convert_to_sparse_model`).\n\nWhat differs, on purpose\n- A present bias is kept and applied unchanged (the official converter drops it with a warning).\n- The pretrained weight is a frozen buffer-like Parameter (requires_grad False) in its loaded dtype; the mask is\n  computed in FP32 and multiplied in the weight's dtype.\n- `bake()` writes W * 1[G > 0] into plain nn.Linear layers so any Hugging Face loader evaluates the hard mask\n  (multiplying by exactly 1.0 or 0.0 is exact, so baked logits equal the in-memory eval-mode logits bit for bit).\n- Saved masks: numpy.packbits + zlib (stdlib) instead of zstd; the size is reported as such.\n- Gate initialisation draws from an explicit CPU torch.Generator, so every backend starts from the same logits.\n\"\"\"\nfrom __future__ import annotations\n\nimport math\nimport re\nimport zlib\nfrom typing import Iterable\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nOFFICIAL = {\"repo\": \"agentic-learning-ai-lab/deleting-weights\", \"commit\": \"94f0340ebb574f95bb41d844165253a6785e215e\",\n            \"licence\": \"MIT\"}\n_no_grad = torch.no_grad  # undotted decorator alias: the lab's older PR guard reads \"@torch.x\" as an e-mail address\nTARGET_SETS = {\n    \"gate\": (\"gate_proj\",),\n    \"mlp\": (\"gate_proj\", \"up_proj\", \"down_proj\"),\n    \"attn\": (\"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\"),\n    # Team 5 LanguageModel (teams/05-architecture/harness/week2/model.py): experts are nn.ModuleList of SwiGLU with\n    # children gate/up/down (routed `moe.experts.N`, shared `moe.shared.N`); attention children q/k/v/o. Substrings\n    # would over-match (\"o\" is in \"blocks\"; \".gate\" is also the optional attention gate), so these are anchored\n    # regular expressions (\"re:\" prefix, full-name match).\n    \"g5_gate\": (r\"re:blocks\\.\\d+\\.moe\\.(experts|shared)\\.\\d+\\.gate\",),\n    \"g5_mlp\": (r\"re:blocks\\.\\d+\\.moe\\.(experts|shared)\\.\\d+\\.(gate|up|down)\",),\n    \"g5_attn\": (r\"re:blocks\\.\\d+\\.attn\\.(q|k|v|o)\",),\n}\n\n\ndef target_match(full: str, targets: Iterable[str]) -> bool:\n    \"\"\"A target is a substring of the module name, or (prefix \"re:\") a regular expression matching the whole name.\"\"\"\n    return any(re.fullmatch(t[3:], full) if t.startswith(\"re:\") else t in full for t in targets)\n\n\nclass MaskedLinear(nn.Module):\n    \"\"\"y = x (W * m)^T + b with W, b frozen. m = sigmoid(alpha G) in training mode, 1[G > 0] in eval mode.\n\n    `.weight` is the EFFECTIVE weight W * m (a property, recomputed on every read; raw W while `unmasked`), and\n    `forward` is F.linear(x, self.weight, self.bias). Code that reads a layer's weight instead of calling it (G5's\n    fused MoE path since #521 stacks `e.gate.weight` of every routed expert) therefore sees the mask, and the mask\n    logits get gradient through it. The frozen pretrained tensor is still the registered parameter named `weight`\n    (so parameters(), named_parameters(), state_dict(), load_state_dict() and every old checkpoint are unchanged) and\n    is read as `base_weight`. Do not assign `layer.weight = ...` (no setter); change `layer.base_weight` in place.\"\"\"\n\n    def __init__(self, base: nn.Linear, alpha_start: float, alpha_end: float, total_steps: int,\n                 generator: torch.Generator):\n        super().__init__()\n        if alpha_start <= 0 or alpha_end <= 0 or total_steps <= 0:\n            raise ValueError(\"alpha_start, alpha_end and total_steps must be positive\")\n        self.in_features, self.out_features = base.in_features, base.out_features\n        self._parameters[\"weight\"] = base.weight  # registered as `weight`; the class property below shadows reads\n        base.weight.requires_grad_(False)\n        self.bias = base.bias\n        if self.bias is not None:\n            self.bias.requires_grad_(False)\n        g = torch.empty(self.base_weight.shape, dtype=torch.float32)\n        g.normal_(0.0, 0.01, generator=generator)\n        self.gate = nn.Parameter(g.to(self.base_weight.device))  # FP32 logits, one per weight\n        self.alpha_start, self.alpha_end, self.total_steps = float(alpha_start), float(alpha_end), int(total_steps)\n        self.current_step = 0\n        self.unmasked = False  # True only during Wanda calibration (the official `sparse = False`)\n\n    @property\n    def alpha(self) -> float:\n        p = min(self.current_step / self.total_steps, 1.0)\n        return self.alpha_start + (self.alpha_end - self.alpha_start) * p\n\n    def step(self) -> None:\n        self.current_step += 1\n\n    def soft_mask(self) -> torch.Tensor:\n        return torch.sigmoid(self.gate * self.alpha)\n\n    def hard_mask(self) -> torch.Tensor:\n        return (self.gate > 0).to(torch.float32)\n\n    def mask(self) -> torch.Tensor:\n        return self.soft_mask() if self.training else self.hard_mask()\n\n    @property\n    def base_weight(self) -> torch.Tensor:\n        \"\"\"The frozen pretrained W (the registered parameter `weight`).\"\"\"\n        return self._parameters[\"weight\"]\n\n    @property\n    def weight(self) -> torch.Tensor:\n        \"\"\"The weight this layer applies: W * m (m from mask(): soft in training mode, hard in eval mode), or the raw\n        frozen W while `unmasked` is set for Wanda calibration. The only place the effective weight is computed.\"\"\"\n        if self.unmasked:\n            return self.base_weight\n        return self.base_weight * self.mask().to(self.base_weight.dtype)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return F.linear(x, self.weight, self.bias)\n\n    def extra_repr(self) -> str:\n        return f\"in_features={self.in_features}, out_features={self.out_features}, bias={self.bias is not None}\"\n\n\ndef masked_layers(model: nn.Module) -> list[tuple[str, MaskedLinear]]:\n    return [(n, m) for n, m in model.named_modules() if isinstance(m, MaskedLinear)]\n\n\ndef convert(model: nn.Module, targets: Iterable[str], alpha_start: float = 100.0, alpha_end: float = 1000.0,\n            total_steps: int = 400, seed: int = 0) -> list[str]:\n    \"\"\"Freeze every parameter, then replace each nn.Linear whose full name contains a target substring by a\n    MaskedLinear. Never touches embeddings, the LM head or norms. Returns the converted names (model order).\"\"\"\n    targets = tuple(targets)\n    if not targets:\n        raise ValueError(\"no DT targets\")\n    for p in model.parameters():\n        p.requires_grad_(False)\n    gen = torch.Generator().manual_seed(seed)\n    names = []\n    for parent_name, parent in list(model.named_modules()):\n        for child_name, child in list(parent.named_children()):\n            full = f\"{parent_name}.{child_name}\" if parent_name else child_name\n            if isinstance(child, nn.Linear) and target_match(full, targets):\n                setattr(parent, child_name, MaskedLinear(child, alpha_start, alpha_end, total_steps, gen))\n                names.append(full)\n    if not names:\n        raise ValueError(f\"no nn.Linear matched {targets}\")\n    return names\n\n\ndef step_all(model: nn.Module) -> None:\n    for _, m in masked_layers(model):\n        m.step()\n\n\ndef regulariser(model: nn.Module) -> torch.Tensor:\n    \"\"\"sum over masked layers of ||W * soft_mask||^2 (FP32), the term the loss subtracts with weight lambda_t.\n    Training calls regulariser_backward_ (same gradient, no graph); this autograd form is kept as its reference.\"\"\"\n    total = None\n    for _, m in masked_layers(model):\n        t = (m.base_weight.detach().float() * m.soft_mask()).pow(2).sum()\n        total = t if total is None else total + t\n    return total\n\n\n@_no_grad()\ndef regulariser_backward_(model: nn.Module, lam_t: float) -> float:\n    \"\"\"Add the gradient of `-lam_t * regulariser(model)` to every logit's .grad, one layer at a time, and return\n    regulariser(model) as a float. Same result as `(-lam_t * regulariser(model)).backward()` bit for bit (each step\n    below is the op autograd runs, in its order: pow -> mul -> sigmoid -> mul by alpha), but no autograd graph is kept.\n\n    Why: `regulariser(model)` sums one term per masked layer before a single backward, so autograd holds every layer's\n    sigmoid output and W * mask at once (8 B per masked weight with an FP32-stored W, 12 B with a bf16-stored W, whose\n    FP32 copy is held too). Here only one layer's temporaries exist at a time, so the regulariser's memory no longer\n    grows with the number of masked weights: by that arithmetic about 14.5 GB less at Qwen3-8B (1.81B gate weights,\n    FP32 W) and 22.8 GB less on the rl-loop 7B DT run (1.90B, bf16 W).\n\n    Assumes what `convert` builds: FP32 logits. A frozen logit (requires_grad False) gets no gradient, as in autograd;\n    its term still counts in the returned value.\"\"\"\n    total = None\n    for _, m in masked_layers(model):\n        s = m.soft_mask()                     # sigmoid(G * alpha), FP32\n        wf = m.base_weight.detach().float()   # a copy only when W is stored in bf16, as in the autograd path\n        p = wf * s\n        t = p.pow(2).sum()\n        total = t if total is None else total + t\n        if m.gate.requires_grad:\n            g = (p * 2).mul_(-lam_t)          # pow backward: grad * (2 * p), grad = 1.0 * (-lam_t) in FP32\n            g.mul_(wf)                        # mul backward (p = wf * s): grad * wf\n            g.mul_(1 - s).mul_(s)             # sigmoid backward: grad * (1 - s) * s\n            g.mul_(m.alpha)                   # mul backward (G * alpha): grad * alpha\n            if m.gate.grad is None:\n                m.gate.grad = g.contiguous()  # the parameter's own layout, whatever W's strides are\n            else:\n                m.gate.grad.add_(g)           # AccumulateGrad: grad += new\n            del g\n        del s, wf, p                          # free this layer's temporaries before the next layer's\n    return float(total) if total is not None else 0.0\n\n\ndef lambda_at(step: int, total_steps: int, lam: float) -> float:\n    \"\"\"Official train_sft.py: weight_reg * (1 - iter / iters), decaying linearly to 0.\"\"\"\n    return lam * (1.0 - step / total_steps)\n\n\ndef fused_moe_layers(model: nn.Module) -> list[tuple[str, nn.Module]]:\n    \"\"\"MoE modules (attributes `router`, `experts` ModuleList, `topk`, forward(x, valid)) that hold masked experts.\n    G5's MoE.forward since #521 reads the experts' weights and never calls them, so a forward hook on an expert\n    layer cannot see its input there; the hook goes on the MoE module instead (see wanda_scores).\"\"\"\n    return [(n, m) for n, m in model.named_modules()\n            if hasattr(m, \"router\") and hasattr(m, \"topk\") and isinstance(getattr(m, \"experts\", None), nn.ModuleList)\n            and any(isinstance(c, MaskedLinear) for c in m.experts.modules())]\n\n\ndef g5_expert_choice(mod: nn.Module, scores: torch.Tensor) -> torch.Tensor:\n    \"\"\"The experts G5's MoE.forward chooses for router softmax `scores` [N, E]: the top-k of `scores + router_bias`\n    when its loss-free balancer is on (`balance_gamma != 0`, set from `moe.bias_gamma`), else the top-k of `scores`.\n    The bias changes only which experts are chosen, never the gate values. A G5 model.py from before the balancer\n    has neither attribute, so this is the plain top-k there, as its forward is.\"\"\"\n    if float(getattr(mod, \"balance_gamma\", 0.0) or 0.0) != 0.0:\n        scores = scores + mod.router_bias.to(scores.dtype)\n    return scores.topk(mod.topk, dim=-1).indices\n\n\n@_no_grad()\ndef wanda_scores(model: nn.Module, batches: Iterable[dict], device) -> dict[str, torch.Tensor]:\n    \"\"\"|W_ij| * ||X_j||_2 per masked layer, with the masks switched off (official compute_wanda_scores).\n\n    Routed experts of a fused MoE (never called as modules): after the MoE runs, every masked expert that no hook saw\n    during that call is run on exactly the rows MoE.forward would route to it (router softmax, top-k with the\n    loss-free balancer's router_bias when it is on, see g5_expert_choice; padding rows excluded: the old per-expert\n    loop's gathered rows), so the same per-layer hooks record its inputs, including the hidden input of `down`,\n    computed with the unmasked weights. A MoE that still calls its experts (the old loop)\n    triggers no extra call, so nothing is counted twice; an expert with no routed row gets no score, as before.\"\"\"\n    layers = dict(masked_layers(model))\n    norms: dict[str, torch.Tensor] = {}\n    hooks = []\n    called: set[int] = set()  # id() of the masked layers whose forward hook fired since the current MoE began\n    for name, m in layers.items():\n        m.unmasked = True\n\n        def hook(mod, inp, _out, name=name):\n            called.add(id(mod))\n            x = inp[0].detach().float()\n            mask = _CALIB_STATE.get(\"attn_mask\")\n            # count real tokens only. A [B, T, d] input drops its padding positions; so does a flattened [B*T, d]\n            # input (G5's shared experts run on every flattened row and zero the padding output afterwards). A layer\n            # that receives gathered rows (a routed expert in G5's loop dispatch gets [routed tokens, d], already\n            # restricted to valid positions, so never B*T rows when padding exists) is used as is.\n            if mask is not None and x.shape[:-1] == mask.shape:\n                x = x[mask]\n            elif mask is not None and x.dim() == 2 and x.shape[0] == mask.numel():\n                x = x[mask.reshape(-1)]\n            elif mask is not None and x.dim() != 2:\n                raise ValueError(f\"calibration input of {name} has shape {tuple(x.shape)}; cannot apply a \"\n                                 f\"{tuple(mask.shape)} padding mask\")\n            sq = x.pow(2).reshape(-1, x.shape[-1]).sum(0)\n            norms[name] = sq if name not in norms else norms[name] + sq\n        hooks.append(m.register_forward_hook(hook))\n\n    def moe_pre(mod, args, kwargs):\n        called.clear()\n\n    def moe_post(mod, args, kwargs, _out):\n        x = args[0] if args else kwargs[\"x\"]\n        valid = args[1] if len(args) > 1 else kwargs[\"valid\"]\n        flat = x.detach().reshape(-1, x.shape[-1])\n        active = valid.reshape(-1).bool()\n        with torch.autocast(device_type=x.device.type, enabled=False):\n            scores = F.linear(flat.float(), mod.router.weight.float()).softmax(-1)\n        choice = g5_expert_choice(mod, scores)\n        for e, expert in enumerate(mod.experts):\n            parts = [c for c in expert.modules() if isinstance(c, MaskedLinear)]\n            if not parts or any(id(c) in called for c in parts):\n                continue  # not masked, or the MoE called it itself (old loop)\n            rows = ((choice == e) & active[:, None]).any(-1).nonzero().flatten()\n            if rows.numel():\n                expert(flat[rows])\n\n    for _, moe in fused_moe_layers(model):\n        hooks.append(moe.register_forward_pre_hook(moe_pre, with_kwargs=True))\n        hooks.append(moe.register_forward_hook(moe_post, with_kwargs=True))\n    was_training = model.training\n    model.eval()\n    try:\n        for b in batches:\n            ids, attn = b[\"input_ids\"].to(device), b[\"attention_mask\"].to(device)\n            _CALIB_STATE[\"attn_mask\"] = attn.bool()\n            model(input_ids=ids, attention_mask=attn)\n    finally:\n        _CALIB_STATE.clear()\n        for h in hooks:\n            h.remove()\n        for m in layers.values():\n            m.unmasked = False\n        model.train(was_training)\n    return {n: layers[n].base_weight.detach().float().abs() * norms[n].sqrt().unsqueeze(0) for n in norms}\n\n\n_CALIB_STATE: dict = {}\n\n\n@_no_grad()\ndef apply_wanda_prior(model: nn.Module, scores: dict[str, torch.Tensor], eta: float, prior_strength: float = 3.0,\n                      unseen: str = \"official\") -> dict:\n    \"\"\"Shift each layer's logits by +-std(G)*prior_strength: down for the lowest-`eta` Wanda scores (official\n    wanda_prior_deletions + apply_wanda_prior: threshold = kthvalue(max(1, int(n * eta))), deleted = score < it).\n\n    `unseen` says what a layer that no calibration token reached gets (a routed MoE expert can be idle on the\n    calibration set; a Wanda score needs an input norm, which is then zero for every column):\n      \"official\"  no prior (the official code's behaviour). The logits stay N(0, 0.01^2), so about half the layer's\n                  weights start deleted under the hard mask, far above `eta`.\n      \"keep\"      shift the whole layer up by std(G)*prior_strength to reduce initial deletion. Logits that remain\n                  nonpositive still delete their weights under the hard mask. In the capstone rehearsal, 30 of 204\n                  expert layers were uncalibrated and the shift reduced overall initial deletion from 11.7% to 4.4%.\"\"\"\n    if unseen not in (\"official\", \"keep\"):\n        raise ValueError(f\"unseen must be official or keep, not {unseen!r}\")\n    marked = 0\n    total = 0\n    unseen_layers = 0\n    for name, m in masked_layers(model):\n        if name not in scores:\n            unseen_layers += 1\n            if unseen == \"keep\":\n                m.gate.add_(m.gate.std().item() * prior_strength)\n            continue  # no calibration token reached the layer: no deletion prior (official behaviour)\n        flat = scores[name].reshape(-1)\n        k = max(1, int(flat.numel() * eta))\n        thr = flat.kthvalue(k).values.item()\n        deleted = (flat < thr).reshape(m.gate.shape)\n        sign = torch.where(deleted, -1.0, 1.0).to(m.gate.device)\n        m.gate.add_(sign * m.gate.std().item() * prior_strength)\n        marked += int(deleted.sum())\n        total += deleted.numel()\n    return {\"eta\": eta, \"prior_strength\": prior_strength, \"prior_deleted\": marked, \"targeted\": total,\n            \"layers_without_scores\": unseen_layers, \"unseen_policy\": unseen}\n\n\n@_no_grad()\ndef deletion_stats(model: nn.Module) -> dict:\n    \"\"\"Realised deletion from the hard mask, over the targeted weights and over every stored parameter.\"\"\"\n    deleted = targeted = 0\n    per_layer = {}\n    for name, m in masked_layers(model):\n        d = int((m.gate <= 0).sum())\n        deleted += d\n        targeted += m.gate.numel()\n        per_layer[name] = d / m.gate.numel()\n    seen, total = set(), 0\n    for n, p in model.named_parameters():\n        if p.data_ptr() in seen or n.endswith(\".marker_rows\"):\n            continue  # tied embeddings counted once; the bench's marker rows are rows of the embedding / LM head\n        seen.add(p.data_ptr())\n        total += p.numel()\n    total -= sum(m.gate.numel() for _, m in masked_layers(model))  # logits are not model weights\n    return {\"deleted\": deleted, \"targeted\": targeted, \"model_weights\": total,\n            \"deleted_frac_of_targeted\": deleted / targeted, \"deleted_frac_of_all_weights\": deleted / total,\n            \"per_layer_min\": min(per_layer.values()), \"per_layer_max\": max(per_layer.values())}\n\n\n@_no_grad()\ndef keep_masks(model: nn.Module) -> list[torch.Tensor]:\n    \"\"\"The hard keep mask 1[G > 0] of every masked layer (bool, on the logits' device), in model order.\"\"\"\n    return [m.gate > 0 for _, m in masked_layers(model)]\n\n\n@_no_grad()\ndef flip_counts(model: nn.Module, prev: list[torch.Tensor]) -> tuple[dict, list[torch.Tensor]]:\n    \"\"\"Gross hard-mask changes since `prev` (from keep_masks): weights newly deleted (kept -> deleted) and restored\n    (deleted -> kept). The net change of the deleted count hides flips in both directions; these do not.\n    Returns (counts, the current keep masks to pass next time). One host sync.\"\"\"\n    now = keep_masks(model)\n    if len(now) != len(prev):\n        raise ValueError(\"prev does not match the masked layers\")\n    newly = torch.stack([(p & ~k).sum() for p, k in zip(prev, now)]).sum()\n    back = torch.stack([(~p & k).sum() for p, k in zip(prev, now)]).sum()\n    a, b = torch.stack([newly, back]).tolist()\n    return {\"newly_deleted\": int(a), \"restored\": int(b)}, now\n\n\n@_no_grad()\ndef band_stats(model: nn.Module, edge: float = 1e-3) -> dict:\n    \"\"\"Where the mask can still learn, measured on all logits (exact, not sampled):\n    live_share: share of logits whose FP32 local derivative sigmoid'(alpha G) is nonzero (outside it the gradient is\n                exactly zero in FP32, so only Adam momentum can still move the logit);\n    band_share: share whose soft mask lies in (edge, 1 - edge), i.e. |alpha G| < log((1 - edge) / edge);\n    soft_hard_gap: mean |sigmoid(alpha G) - 1[G > 0]| over the targeted weights, the per-weight distance between the\n                   trainer's soft mask and the sampler's hard mask. One host sync.\"\"\"\n    live, band, gap, n = [], [], [], 0\n    for _, m in masked_layers(model):\n        s = m.soft_mask()\n        live.append((s * (1 - s) > 0).sum())\n        band.append(((s > edge) & (s < 1 - edge)).sum())\n        gap.append((s - (m.gate > 0).to(s.dtype)).abs().sum(dtype=torch.float64))\n        n += m.gate.numel()\n    lv, bd = torch.stack(live).sum(), torch.stack(band).sum()\n    g = torch.stack(gap).sum()\n    lv, bd, g = torch.stack([lv.double(), bd.double(), g]).tolist()\n    return {\"live_share\": lv / n, \"band_share\": bd / n, \"soft_hard_gap\": g / n}\n\n\n@_no_grad()\ndef bake(model: nn.Module) -> dict[str, torch.Tensor]:\n    \"\"\"Replace every MaskedLinear by nn.Linear(W * 1[G > 0], b) in place. Returns {name.weight: bool keep mask}.\"\"\"\n    masks = {}\n    for parent_name, parent in list(model.named_modules()):\n        for child_name, child in list(parent.named_children()):\n            if isinstance(child, MaskedLinear):\n                full = f\"{parent_name}.{child_name}\" if parent_name else child_name\n                keep = child.gate > 0\n                lin = nn.Linear(child.in_features, child.out_features, bias=child.bias is not None,\n                                device=child.base_weight.device, dtype=child.base_weight.dtype)\n                lin.weight.copy_(child.base_weight * keep.to(child.base_weight.dtype))\n                if child.bias is not None:\n                    lin.bias.copy_(child.bias)\n                lin.requires_grad_(False)\n                setattr(parent, child_name, lin)\n                masks[f\"{full}.weight\"] = keep.cpu()\n    return masks\n\n\ndef pack_masks(masks: dict[str, torch.Tensor]) -> tuple[bytes, dict]:\n    \"\"\"One bit per weight (numpy.packbits), zlib level 9 over the concatenated bit streams in sorted name order.\"\"\"\n    names = sorted(masks)\n    raw = b\"\".join(np.packbits(masks[n].numpy().ravel()).tobytes() for n in names)\n    comp = zlib.compress(raw, 9)\n    n_bits = sum(masks[n].numel() for n in names)\n    return comp, {\"format\": \"packbits+zlib9\", \"layers\": len(names), \"bits\": n_bits, \"packed_bytes\": len(raw),\n                  \"compressed_bytes\": len(comp), \"shapes\": {n: list(masks[n].shape) for n in names}}\n\n\ndef unpack_masks(blob: bytes, meta: dict) -> dict[str, torch.Tensor]:\n    raw = zlib.decompress(blob)\n    out, off = {}, 0\n    for n in sorted(meta[\"shapes\"]):\n        shape = meta[\"shapes\"][n]\n        k = math.prod(shape)\n        nb = (k + 7) // 8\n        bits = np.unpackbits(np.frombuffer(raw[off:off + nb], dtype=np.uint8))[:k]\n        out[n] = torch.from_numpy(bits.astype(bool)).reshape(shape)\n        off += nb\n    return out\n"
}
# dt_on_quantized.py reads ../10-post-training/dt/mask.py: recreate that layout
os.makedirs(f"{WORK}/teams/12-quantization/dt_quant", exist_ok=True); os.makedirs(f"{WORK}/teams/10-post-training/dt", exist_ok=True)
open(f"{WORK}/teams/12-quantization/dt_quant/dt_on_quantized.py", "w").write(SOURCES["dt_on_quantized.py"])
open(f"{WORK}/teams/10-post-training/dt/mask.py", "w").write(SOURCES["mask.py"])
CODE = f"{WORK}/teams/12-quantization/dt_quant"
print("wrote", sorted(SOURCES), "under", WORK)

In [ ]:
# a venv with pinned packages (Kaggle's own torch can fail to import); GPU must be visible to it
VENV = f"{WORK}/venv"; PY = f"env -u PYTHONPATH {VENV}/bin/python"
if not os.path.exists(f"{VENV}/.ready"):
    sh(f"rm -rf {VENV} && pip install -q uv==0.13.6 && uv venv -q --python python3 {VENV} && "
       f"uv pip install -q --python {VENV}/bin/python {TORCH} --index-url {TORCH_INDEX} && "
       f"uv pip install -q --python {VENV}/bin/python transformers==4.57.6 pyarrow numpy accelerate && touch {VENV}/.ready",
       log=f"{OUT}/logs/venv.log")
probe = subprocess.run(f"{PY} -c \"import torch, transformers, pyarrow, json; print(json.dumps(dict(torch=torch.__version__, "
                       f"cuda=torch.cuda.is_available(), transformers=transformers.__version__, pyarrow=pyarrow.__version__)))\"",
                       shell=True, capture_output=True, text=True)
if probe.returncode != 0:
    raise RuntimeError("the venv does not work: " + probe.stderr[-3000:])
ENV["packages"] = json.loads(probe.stdout.strip().splitlines()[-1]); print(ENV["packages"])
if not ENV["packages"]["cuda"]:
    raise RuntimeError("torch cannot see the GPU: set TORCH_INDEX to https://download.pytorch.org/whl/cu128 and delete "
                       f"{VENV}, then rerun")
json.dump(ENV, open(f"{OUT}/environment.json", "w"), indent=1)

In [ ]:
# the run: one result file; arms already in it are skipped, so a stopped session resumes
RUN = f"{OUT}/dt_quant_{MODEL.split('/')[-1]}_s{STEPS}_w{TRAIN_WINDOWS}_lr{LR:g}"
os.makedirs(RUN, exist_ok=True)
RES = f"{RUN}/results.json"
rev = f"--revision {REVISION} " if REVISION else ""
sh(f"cd {CODE} && env HF_HOME={WORK}/hf {PY} -u dt_on_quantized.py --model {MODEL} {rev}--out {RES} --arms {','.join(ARMS)} "
   f"--steps {STEPS} --batch {BATCH} --micro {MICRO} --length {LENGTH} --train-windows {TRAIN_WINDOWS} "
   f"--test-windows {TEST_WINDOWS} --lr {LR} --seed {SEED} --device cuda --grad-check", log=f"{OUT}/logs/run.log")

In [ ]:
# results: one table, a zip of everything
import pandas as pd
d = json.load(open(RES))
rows = []
for name, r in d["arms"].items():
    m = r["memory"]
    rows.append({"arm": name, "nll": round(r["test_nll_mean"], 4), "deleted_%": round(100 * r["deletion_final"], 3) if "deletion_final" in r else None,
                 "prior_deleted": r["wanda_prior"]["prior_deleted"] if "wanda_prior" in r else None,
                 "init_deleted_%": round(100 * r["deletion_at_init"], 3) if "deletion_at_init" in r else None,
                 "train_s": r.get("train_seconds"), "peak_cuda_GB": r.get("peak_cuda_gb"),
                 "decoder_weights_GB": round(m["weights_bytes"] / 1e9, 3), "dt_state_GB": round(m["dt_state_bytes"] / 1e9, 3),
                 "packed_GB_targeted": round(r["packed_bytes_targeted"] / 1e9, 3) if "packed_bytes_targeted" in r else None})
table = pd.DataFrame(rows); table.to_csv(f"{RUN}/table.csv", index=False)
print(table.to_string(index=False))
for q, r in d.get("grad_coverage", {}).items():
    print(f"gradient coverage, {q}: packed path on - {r['fast_path_on']['trainable_parameters']} trainable, none {len(r['fast_path_on']['grad_none'])}, "
          f"all zero {len(r['fast_path_on']['grad_all_zero'])}; off - none {len(r['fast_path_off']['grad_none'])}, all zero {len(r['fast_path_off']['grad_all_zero'])}; "
          f"same lists {r['same_none_and_zero_lists']}, max gradient difference {r['max_abs_grad_difference']}")
print("\npaired NLL differences (mean over windows, 95% interval):")
for k, v in d.get("paired", {}).items():
    print(f"  {k}: {v['mean_diff']:+.4f}  [{v['ci95'][0]:+.4f}, {v['ci95'][1]:+.4f}]  ({v['windows']} windows)")
ENV["finished"] = datetime.datetime.now().isoformat(timespec="seconds")
json.dump(ENV, open(f"{OUT}/environment.json", "w"), indent=1)
shutil.copy(f"{OUT}/environment.json", f"{RUN}/environment.json"); shutil.copytree(f"{OUT}/logs", f"{RUN}/logs", dirs_exist_ok=True)
zip_path = shutil.make_archive(RUN, "zip", RUN)
print("\nEverything is in", RUN, "and", zip_path, "(Kaggle: the version's Output tab)")